# Reconstruction Notebook Documentation

This notebook turns a room video into a Nerfstudio-compatible reconstruction dataset, trains a Gaussian splat model, and exports the final splat artifact.

It is designed for Google Colab with a Tesla T4 GPU. The notebook uses Google Drive for durable storage and /content for fast temporary work. Anything under /content is lost when the Colab runtime is restarted or switched between CPU and GPU, so the notebook syncs important outputs back to Drive after each major stage.

## Main Outputs

- selected reconstruction frames: /content/drive/MyDrive/room-model-project/scans/scan_001/frames/selected_clean
- anchor frame list: /content/drive/MyDrive/room-model-project/scans/scan_001/frames/anchor_frames.txt
- anchor or loop pair list: /content/drive/MyDrive/room-model-project/scans/scan_001/frames/anchor_pairs.txt
- COLMAP model: /content/drive/MyDrive/room-model-project/scans/scan_001/processed/colmap
- Nerfstudio dataset: /content/drive/MyDrive/room-model-project/scans/scan_001/nerfstudio-data
- training outputs: /content/drive/MyDrive/room-model-project/scans/scan_001/training
- exported splat: /content/drive/MyDrive/room-model-project/scans/scan_001/exports

## Recommended Colab Runtime

Use a GPU runtime for this notebook, preferably a Tesla T4. CPU can run setup and some inspection cells, but training and GPU COLMAP are expected to be much slower or unavailable on CPU.

## 1. Mount Google Drive

This cell connects Colab to your Drive project folder. Run it once at the start of a fresh runtime. If Colab asks for authorization, approve the Drive mount before continuing.

The notebook expects the project to live at:

/content/drive/MyDrive/room-model-project/scans/scan_001

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Select Clean Reconstruction Frames

This cell reads the source room video, selects a smaller set of sharp and visually useful frames, and writes those frames to local /content first for speed. At the end, it syncs the selected frames and pair lists back to Drive.

The selection has three goals:

- continuity frames keep enough coverage along the camera path
- anchor frames preserve high-feature views that help reconstruction stay stable
- loop pair frames help COLMAP reconnect views from different points in the scan

### Important Tuning Knobs

- PROCESS_EVERY_N_FRAMES controls how many video frames are skipped while scanning. Higher is faster, but may lose coverage.
- MAX_TOTAL_FRAMES controls the final frame budget. Lower is faster for COLMAP and training. Higher can improve quality if the scan needs more coverage.
- MAX_ANCHOR_FRAMES and MAX_LOOP_CONTINUITY_CANDIDATES affect loop closure quality and matching cost.
- BLUR_THRESHOLD filters out blurry frames. If too few frames are selected, lower it.

Current defaults are tuned for faster Colab iteration rather than maximum final quality.

In [ ]:
import cv2
import os
import shutil
import subprocess
import numpy as np
from collections import deque

# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/room-model-project/scans/scan_001"
LOCAL_PROJECT_DIR = "/content/room-model-project/scans/scan_001"

DRIVE_VIDEO_PATH = f"{DRIVE_PROJECT_DIR}/input/video.MOV"
VIDEO_PATH = f"{LOCAL_PROJECT_DIR}/input/video.MOV"

OUT_DIR = f"{LOCAL_PROJECT_DIR}/frames/selected_clean"
DRIVE_OUT_DIR = f"{DRIVE_PROJECT_DIR}/frames/selected_clean"

ANCHOR_FILE = f"{LOCAL_PROJECT_DIR}/frames/anchor_frames.txt"
DRIVE_ANCHOR_FILE = f"{DRIVE_PROJECT_DIR}/frames/anchor_frames.txt"

ANCHOR_PAIRS_FILE = f"{LOCAL_PROJECT_DIR}/frames/anchor_pairs.txt"
DRIVE_ANCHOR_PAIRS_FILE = f"{DRIVE_PROJECT_DIR}/frames/anchor_pairs.txt"

os.makedirs(os.path.dirname(VIDEO_PATH), exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(ANCHOR_FILE), exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if not os.path.exists(VIDEO_PATH) or os.path.getsize(VIDEO_PATH) != os.path.getsize(DRIVE_VIDEO_PATH):
    subprocess.run(["rsync", "-a", "--info=progress2", DRIVE_VIDEO_PATH, os.path.dirname(VIDEO_PATH) + "/"], check=True)

shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# PARAMETERS
# ============================================================

# Process every Nth video frame.
# Your video is ~60 FPS, so processing every frame is wasteful.
PROCESS_EVERY_N_FRAMES = 3

# Continuity frame selection.
MIN_FRAME_GAP = 6
MAX_FRAME_GAP = 28

DIFF_MIN = 6.5
DIFF_MAX = 45.0

BLUR_THRESHOLD = 80.0
RECENT_MEMORY = 5

# Final output frame target.
MAX_TOTAL_FRAMES = 180

# Anchor frame selection.
MAX_ANCHOR_FRAMES = 25
ANCHOR_MIN_DISTANCE = 40
ANCHOR_SIMILARITY_THRESHOLD = 10.0

# Loop candidate selection.
MAX_LOOP_CONTINUITY_CANDIDATES = 60
LOOP_CANDIDATE_MIN_FRAME_DISTANCE = 35
LOOP_CANDIDATE_VISUAL_DISTANCE = 8.0

# Loop pair detection.
MAX_LOOP_PAIRS = 20
MIN_LOOP_TEMPORAL_DISTANCE = 120
MIN_GOOD_MATCHES = 30
LOOP_MATCH_DISTANCE = 50


# ============================================================
# ORB SETUP
# ============================================================

# ORB is used only for final loop-pair matching.
# Do not extract ORB descriptors for every frame.
orb = cv2.ORB_create(
    nfeatures=800,
    scaleFactor=1.2,
    nlevels=6
)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)


# ============================================================
# SCORING FUNCTIONS
# ============================================================

def blur_score(gray):
    """
    Measures image sharpness.
    Higher = sharper.
    """
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def diff_score(a, b):
    """
    Fast visual difference between two small grayscale frames.
    Lower = more similar.
    Higher = more different.
    """
    return np.mean(cv2.absdiff(a, b))


def fast_feature_score(gray):
    """
    Fast feature-richness estimate.

    This is cheaper than ORB and is used for ranking anchors
    and loop candidates.
    """
    corners = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=500,
        qualityLevel=0.01,
        minDistance=5
    )

    return 0 if corners is None else len(corners)


def extract_orb(gray):
    """
    Extract ORB descriptors for robust loop matching.

    This is only called on the reduced loop_pool, not all frames.
    """
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    return keypoints, descriptors


def loop_match_score(desc_a, desc_b):
    """
    Returns number of good ORB matches between two frames.

    Higher = stronger loop-closure candidate.
    """
    if desc_a is None or desc_b is None:
        return 0

    if len(desc_a) == 0 or len(desc_b) == 0:
        return 0

    matches = bf.match(desc_a, desc_b)
    good = [m for m in matches if m.distance <= LOOP_MATCH_DISTANCE]

    return len(good)


# ============================================================
# LOAD VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

print("Video exists:", os.path.exists(VIDEO_PATH))
print("Video opened:", cap.isOpened())
print("Total video frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

if not cap.isOpened():
    raise RuntimeError("OpenCV could not open video. Check path or convert MOV to MP4.")


# ============================================================
# PASS 1: READ LIGHTWEIGHT CANDIDATES
# ============================================================

all_candidates = []
frame_id = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Skip frames to reduce runtime.
    if frame_id % PROCESS_EVERY_N_FRAMES != 0:
        frame_id += 1
        continue

    # Small image is used only for scoring/comparison.
    small = cv2.resize(frame, (320, 180))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

    sharpness = blur_score(gray)

    if sharpness >= BLUR_THRESHOLD:
        feat = fast_feature_score(gray)

        all_candidates.append({
            "frame_id": frame_id,
            "gray": gray.copy(),
            "sharpness": sharpness,
            "features": feat,
            "descriptors": None,
            "type": set()
        })

    frame_id += 1

cap.release()

print("Sharp candidate frames:", len(all_candidates))

if len(all_candidates) == 0:
    raise RuntimeError("No frames passed blur threshold. Lower BLUR_THRESHOLD.")


# ============================================================
# PASS 2: SELECT CONTINUITY FRAMES
# ============================================================

continuity = []
recent_grays = deque(maxlen=RECENT_MEMORY)

last_saved_gray = None
last_saved_frame_id = -10**9

for item in all_candidates:
    gray = item["gray"]
    fid = item["frame_id"]

    frames_since_last = fid - last_saved_frame_id

    if last_saved_gray is None:
        should_save = True
    else:
        diff_last = diff_score(gray, last_saved_gray)

        too_soon = frames_since_last < MIN_FRAME_GAP
        too_late = frames_since_last >= MAX_FRAME_GAP

        enough_change = diff_last >= DIFF_MIN
        not_massive_jump = diff_last <= DIFF_MAX

        similar_to_recent = False

        for old_gray in recent_grays:
            if diff_score(gray, old_gray) < DIFF_MIN:
                similar_to_recent = True
                break

        should_save = (
            (
                not too_soon
                and enough_change
                and not_massive_jump
                and not similar_to_recent
            )
            or too_late
        )

    if should_save:
        item["type"].add("continuity")
        continuity.append(item)

        last_saved_gray = gray.copy()
        last_saved_frame_id = fid
        recent_grays.append(gray.copy())

print("Continuity frames:", len(continuity))


# ============================================================
# PASS 3: SELECT GLOBAL ANCHOR FRAMES
# ============================================================

anchor_candidates = sorted(
    all_candidates,
    key=lambda x: (x["features"], x["sharpness"]),
    reverse=True
)

anchors = []

for item in anchor_candidates:
    if len(anchors) >= MAX_ANCHOR_FRAMES:
        break

    fid = item["frame_id"]
    gray = item["gray"]

    reject = False

    for a in anchors:
        if abs(fid - a["frame_id"]) < ANCHOR_MIN_DISTANCE:
            reject = True
            break

        if diff_score(gray, a["gray"]) < ANCHOR_SIMILARITY_THRESHOLD:
            reject = True
            break

    if reject:
        continue

    item["type"].add("anchor")
    anchors.append(item)

print("Anchor frames:", len(anchors))


# ============================================================
# PASS 4: SMART LOOP-CLOSURE PAIR DETECTION
# ============================================================

loop_pairs = []

# Select feature-rich, temporally spread, visually diverse
# continuity frames for loop comparison.
continuity_ranked = sorted(
    continuity,
    key=lambda x: (x["features"], x["sharpness"]),
    reverse=True
)

selected_loop_continuity = []

for item in continuity_ranked:
    if len(selected_loop_continuity) >= MAX_LOOP_CONTINUITY_CANDIDATES:
        break

    fid = item["frame_id"]
    gray = item["gray"]

    reject = False

    for chosen in selected_loop_continuity:
        if abs(fid - chosen["frame_id"]) < LOOP_CANDIDATE_MIN_FRAME_DISTANCE:
            reject = True
            break

        if diff_score(gray, chosen["gray"]) < LOOP_CANDIDATE_VISUAL_DISTANCE:
            reject = True
            break

    if reject:
        continue

    selected_loop_continuity.append(item)

# Build reduced loop comparison pool.
loop_pool_by_id = {}

for item in anchors + selected_loop_continuity:
    loop_pool_by_id[item["frame_id"]] = item

loop_pool = sorted(loop_pool_by_id.values(), key=lambda x: x["frame_id"])

print("Selected loop-continuity candidates:", len(selected_loop_continuity))
print("Loop comparison pool:", len(loop_pool))

# Extract ORB only for loop_pool frames.
for item in loop_pool:
    keypoints, descriptors = extract_orb(item["gray"])
    item["descriptors"] = descriptors
    item["features"] = 0 if keypoints is None else len(keypoints)

# Match far-apart loop candidates.
for i in range(len(loop_pool)):
    if i % 10 == 0:
        print(f"Loop matching progress: {i}/{len(loop_pool)}")

    a = loop_pool[i]

    for j in range(i + 1, len(loop_pool)):
        b = loop_pool[j]

        temporal_gap = abs(b["frame_id"] - a["frame_id"])

        if temporal_gap < MIN_LOOP_TEMPORAL_DISTANCE:
            continue

        good_matches = loop_match_score(a["descriptors"], b["descriptors"])

        if good_matches >= MIN_GOOD_MATCHES:
            loop_pairs.append({
                "a": a,
                "b": b,
                "good_matches": good_matches,
                "temporal_gap": temporal_gap
            })

loop_pairs = sorted(loop_pairs, key=lambda x: x["good_matches"], reverse=True)
loop_pairs = loop_pairs[:MAX_LOOP_PAIRS]

loop_frames = []

for pair in loop_pairs:
    pair["a"]["type"].add("loop")
    pair["b"]["type"].add("loop")

    loop_frames.append(pair["a"])
    loop_frames.append(pair["b"])

print("Loop pairs:", len(loop_pairs))
print("Loop frames before dedupe:", len(loop_frames))


# ============================================================
# PASS 5: MERGE ALL FRAME TYPES
# ============================================================

merged_by_id = {}

for item in continuity + anchors + loop_frames:
    merged_by_id[item["frame_id"]] = item

merged = sorted(merged_by_id.values(), key=lambda x: x["frame_id"])

print("Merged frames before cap:", len(merged))


# ============================================================
# PASS 6: DIVERSITY-AWARE CAP
# ============================================================

must_keep = []
optional = []

for item in merged:
    if "anchor" in item["type"] or "loop" in item["type"]:
        must_keep.append(item)
    else:
        optional.append(item)

remaining_slots = MAX_TOTAL_FRAMES - len(must_keep)

if remaining_slots <= 0:
    must_keep = sorted(
        must_keep,
        key=lambda x: (
            1 if "loop" in x["type"] else 0,
            1 if "anchor" in x["type"] else 0,
            x["features"],
            x["sharpness"]
        ),
        reverse=True
    )[:MAX_TOTAL_FRAMES]

    final = must_keep

else:
    if len(optional) > remaining_slots:
        capped_optional = []
        group_size = len(optional) / remaining_slots

        previous_gray = None

        for i in range(remaining_slots):
            start = int(i * group_size)
            end = int((i + 1) * group_size)

            if end <= start:
                end = start + 1

            group = optional[start:end]

            best_item = None
            best_score = -1e18

            for item in group:
                gray = item["gray"]

                if previous_gray is None:
                    diff_prev = DIFF_MIN
                else:
                    diff_prev = diff_score(gray, previous_gray)

                score = (
                    2.0 * diff_prev
                    + 0.01 * item["features"]
                    + 0.001 * item["sharpness"]
                )

                if diff_prev > DIFF_MAX:
                    score -= (diff_prev - DIFF_MAX) * 2.0

                if score > best_score:
                    best_score = score
                    best_item = item

            capped_optional.append(best_item)
            previous_gray = best_item["gray"]

        final = must_keep + capped_optional

    else:
        final = must_keep + optional

final = sorted(final, key=lambda x: x["frame_id"])

print("Final selected frames:", len(final))


# ============================================================
# PASS 7: CLEAN OUTPUT DIRECTORY
# ============================================================

for f in os.listdir(OUT_DIR):
    if f.lower().endswith((".jpg", ".jpeg", ".png")):
        os.remove(os.path.join(OUT_DIR, f))


# ============================================================
# PASS 8: SAVE FINAL FRAMES USING SECOND VIDEO PASS
# ============================================================

final_frame_ids = set(item["frame_id"] for item in final)

frame_id_to_output_name = {}

cap = cv2.VideoCapture(VIDEO_PATH)

frame_id = 0
save_index = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    if frame_id in final_frame_ids:
        output_name = f"frame_{save_index:05d}.jpg"
        out_path = os.path.join(OUT_DIR, output_name)

        cv2.imwrite(out_path, frame)

        frame_id_to_output_name[frame_id] = output_name
        save_index += 1

    frame_id += 1

cap.release()

print("Actually saved frames:", save_index)


# ============================================================
# PASS 9: SAVE ANCHOR FRAME LIST
# ============================================================

with open(ANCHOR_FILE, "w") as f:
    for item in final:
        if "anchor" in item["type"]:
            f.write(frame_id_to_output_name[item["frame_id"]] + "\n")


# ============================================================
# PASS 10: SAVE LOOP / ANCHOR PAIRS
# ============================================================

with open(ANCHOR_PAIRS_FILE, "w") as f:
    for pair in loop_pairs:
        a_id = pair["a"]["frame_id"]
        b_id = pair["b"]["frame_id"]

        if a_id in frame_id_to_output_name and b_id in frame_id_to_output_name:
            f.write(
                frame_id_to_output_name[a_id]
                + " "
                + frame_id_to_output_name[b_id]
                + "\n"
            )


# ============================================================
# FINAL REPORT
# ============================================================

num_anchors_final = sum("anchor" in item["type"] for item in final)
num_loops_final = sum("loop" in item["type"] for item in final)
num_continuity_final = sum("continuity" in item["type"] for item in final)

print("Saved frames to:", OUT_DIR)
print("Anchor frame list:", ANCHOR_FILE)
print("Anchor/loop pair list:", ANCHOR_PAIRS_FILE)

print("Final continuity frames:", num_continuity_final)
print("Final anchor frames:", num_anchors_final)
print("Final loop frames:", num_loops_final)
print("Final total frames:", len(final))

# Persist selected frames and pair lists back to Drive before changing runtime.
shutil.rmtree(DRIVE_OUT_DIR, ignore_errors=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", OUT_DIR + "/", DRIVE_OUT_DIR + "/"], check=True)
shutil.copy2(ANCHOR_FILE, DRIVE_ANCHOR_FILE)
shutil.copy2(ANCHOR_PAIRS_FILE, DRIVE_ANCHOR_PAIRS_FILE)
print("Synced selected frames to:", DRIVE_OUT_DIR)

## 3. Install Nerfstudio If Needed

This cell checks whether ns-train already exists in the current runtime. If it does, the install is skipped. This avoids spending several minutes reinstalling Nerfstudio during repeated runs in the same Colab session.

In [ ]:
import shutil
import subprocess
import sys

if shutil.which("ns-train") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nerfstudio"], check=True)
else:
    print("nerfstudio already installed")

## 4. Verify Nerfstudio Command Availability

This prints the ns-train help text. It is a lightweight sanity check that Nerfstudio installed correctly and that the command is available on PATH.

In [ ]:
!ns-train --help

## 5. Install COLMAP and Xvfb If Needed

This cell installs COLMAP only when it is missing. Xvfb is used because COLMAP commands can depend on Qt libraries even when running headless in Colab.

If apt install fails, rerun the cell once. Colab package mirrors occasionally fail transiently.

In [ ]:
import shutil
import subprocess

if shutil.which("colmap") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "colmap"], check=True)
else:
    print("colmap already installed")

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get", "install", "-y", "xvfb", "libqt5gui5", "libqt5widgets5", "libqt5core5a", "libgl1-mesa-glx"], check=True)
else:
    print("xvfb already installed")

## 6. Configure Paths and Runtime Knobs

This cell defines the Drive paths, local /content paths, COLMAP paths, log path, and GPU-related settings.

### GPU Settings for Tesla T4

- COLMAP_USE_GPU = 1 enables GPU SIFT extraction and matching.
- COLMAP_MAX_IMAGE_SIZE = 1600 limits image resolution to reduce memory pressure.
- COLMAP_MAX_NUM_FEATURES = 4096 and COLMAP_MAX_NUM_MATCHES = 4096 keep matching manageable.

If COLMAP reports that CUDA SIFT is unavailable, set COLMAP_USE_GPU to 0 and rerun the COLMAP cells.

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/room-model-project/scans/scan_001"
LOCAL_PROJECT_DIR = "/content/room-model-project/scans/scan_001"

DRIVE_IMAGE_DIR = f"{DRIVE_PROJECT_DIR}/frames/selected_clean"
IMAGE_DIR = f"{LOCAL_PROJECT_DIR}/frames/selected_clean"
DRIVE_NS_DATA_DIR = f"{DRIVE_PROJECT_DIR}/nerfstudio-data"
NS_DATA_DIR = f"{LOCAL_PROJECT_DIR}/nerfstudio-data"
LOCAL_OUTPUT_DIR = "/content/room-wall-output"
DRIVE_OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs"

DRIVE_COLMAP_DIR = f"{DRIVE_PROJECT_DIR}/processed/colmap"
COLMAP_DIR = f"{LOCAL_PROJECT_DIR}/processed/colmap"
DB_DIR = f"{COLMAP_DIR}/database.db"
SPARSE_DIR = f"{COLMAP_DIR}/sparse"
LOG_DIR = f"{DRIVE_PROJECT_DIR}/reports/exhaustive_keyframes_colmap_log.txt"

COLMAP_USE_GPU = 1
COLMAP_MAX_IMAGE_SIZE = 1600
COLMAP_MAX_NUM_FEATURES = 4096
COLMAP_MAX_NUM_MATCHES = 4096

# Runtime controls. Keep reconstruction products unless their inputs or settings changed.
REUSE_EXISTING_COLMAP_DATABASE = True
REUSE_EXISTING_SPARSE_MODEL = True
REUSE_EXISTING_ROOM_TRAINING = True
RUN_OPTIONAL_DEPTH_RENDERING = False
WRITE_DEPTH_VARIANTS = False
WRITE_DEPTH_PREVIEWS = False
print('Runtime reuse:', REUSE_EXISTING_COLMAP_DATABASE, REUSE_EXISTING_SPARSE_MODEL, REUSE_EXISTING_ROOM_TRAINING)


## 7. Stage Frames Locally and Reset COLMAP Work Folder

This copies selected frames from Drive into /content before running COLMAP. Running feature extraction and matching directly against Drive is much slower because it touches many image files and a database.

The local COLMAP folder is cleared before each reconstruction run so stale databases or sparse models do not contaminate the next result.

In [ ]:
!mkdir -p "$(dirname '{LOG_DIR}')"
!mkdir -p "{IMAGE_DIR}"
!rsync -a --info=progress2 "{DRIVE_IMAGE_DIR}/" "{IMAGE_DIR}/"
if not REUSE_EXISTING_COLMAP_DATABASE:
    shutil.rmtree(COLMAP_DIR, ignore_errors=True)
!mkdir -p "{SPARSE_DIR}"

## 8. COLMAP Feature Extraction

This extracts SIFT features for the selected images. On a Tesla T4, GPU SIFT can be much faster than CPU SIFT if the installed COLMAP build supports CUDA.

Common issues:

- If GPU extraction fails, set COLMAP_USE_GPU = 0 in the configuration cell.
- If the cell runs out of memory, lower COLMAP_MAX_IMAGE_SIZE or COLMAP_MAX_NUM_FEATURES.
- If too few images register later, increase MAX_TOTAL_FRAMES or COLMAP_MAX_NUM_FEATURES.

In [ ]:
!xvfb-run -a colmap feature_extractor \
    --database_path "{DB_DIR}" \
    --image_path "{IMAGE_DIR}" \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model SIMPLE_RADIAL \
    --SiftExtraction.use_gpu {COLMAP_USE_GPU} \
    --SiftExtraction.gpu_index 0 \
    --SiftExtraction.max_image_size {COLMAP_MAX_IMAGE_SIZE} \
    --SiftExtraction.max_num_features {COLMAP_MAX_NUM_FEATURES} \
    2>&1 | tee -a "{LOG_DIR}"

## 9. Cache the COLMAP Vocabulary Tree

The vocabulary tree improves loop detection during sequential matching. This cell stores the large file in Drive the first time it is downloaded, then copies it to /content for fast local use in future runtime sessions.

In [ ]:
# Cache COLMAP 32K vocab tree in Drive, then use local /content copy.
DRIVE_VOCAB_TREE = f"{DRIVE_PROJECT_DIR}/cache/vocab_tree_flickr100K_words32K.bin"
VOCAB_TREE = "/content/vocab_tree_flickr100K_words32K.bin"

!mkdir -p "$(dirname '{DRIVE_VOCAB_TREE}')"
!test -s "{DRIVE_VOCAB_TREE}" || wget -O "{DRIVE_VOCAB_TREE}" "https://github.com/colmap/colmap/releases/download/3.11.1/vocab_tree_flickr100K_words32K.bin"
!cp "{DRIVE_VOCAB_TREE}" "{VOCAB_TREE}"

## 10. Sequential Matching With Loop Detection

This matches nearby frames in sequence and uses the vocabulary tree to detect loops. Sequential matching is much faster than exhaustive matching for video-derived frame sets.

The overlap setting controls how many neighboring frames each image tries to match. If reconstruction becomes fragmented, increasing overlap can help but will increase runtime.

In [ ]:
# Make sure log directory exists
!mkdir -p "$(dirname "$LOG_DIR")"

# Faster sequential + vocab-tree loop detection
!xvfb-run -a colmap sequential_matcher \
    --database_path "$DB_DIR" \
    --SiftMatching.use_gpu $COLMAP_USE_GPU \
    --SiftMatching.gpu_index 0 \
    --SiftMatching.guided_matching 0 \
    --SiftMatching.max_num_matches $COLMAP_MAX_NUM_MATCHES \
    --SequentialMatching.overlap 12 \
    --SequentialMatching.quadratic_overlap 1 \
    --SequentialMatching.loop_detection 1 \
    --SequentialMatching.vocab_tree_path "$VOCAB_TREE" \
    2>&1 | tee -a "$LOG_DIR"

## 11. Import Anchor and Loop Pair Matches

This cell imports the selected anchor and loop pairs created during frame selection. It adds targeted long-range matches without running a full exhaustive matcher.

If anchor_pairs.txt is empty, this cell may have little effect, but the sequential matcher can still produce a reconstruction.

In [ ]:
# ------------------------------------------------------------
# Match only the selected anchor / loop pairs
#
# This uses anchor_pairs.txt as a list of image pairs:
#
#   frame_00012.jpg frame_00231.jpg
#   frame_00045.jpg frame_00190.jpg
#
# It does NOT run full exhaustive matching.
# ------------------------------------------------------------

PAIR_FILE=f"{LOCAL_PROJECT_DIR}/frames/anchor_pairs.txt"

!xvfb-run -a colmap matches_importer \
    --database_path "$DB_DIR" \
    --match_list_path "$PAIR_FILE" \
    --match_type pairs \
    --SiftMatching.use_gpu $COLMAP_USE_GPU \
    --SiftMatching.gpu_index 0 \
    --SiftMatching.guided_matching 1 \
    --SiftMatching.max_num_matches $COLMAP_MAX_NUM_MATCHES \
    2>&1 | tee -a "$LOG_DIR"

## 12. Run COLMAP Mapper and Persist Sparse Model

The mapper builds the sparse camera reconstruction from the image database and matches. After mapping, the local COLMAP folder is synced back to Drive so it survives a runtime restart or runtime-type switch.

The expected successful model path is usually sparse/0.

In [ ]:
!xvfb-run -a colmap mapper \
    --database_path "{DB_DIR}" \
    --image_path "{IMAGE_DIR}" \
    --output_path "{SPARSE_DIR}"

!rm -rf "{DRIVE_COLMAP_DIR}"
!mkdir -p "{DRIVE_COLMAP_DIR}"
!rsync -a --info=progress2 "{COLMAP_DIR}/" "{DRIVE_COLMAP_DIR}/"

## 13. Inspect COLMAP Output Files

This lists the files created by COLMAP. A healthy sparse model should include cameras, images, and points3D files under sparse/0.

In [ ]:
!find "{SPARSE_DIR}" -maxdepth 3 -type f | sort

## 14. Analyze the Sparse Model

This prints COLMAP model statistics. Check the number of registered images. If many selected frames are missing, the scan may need more overlap, sharper frames, or less aggressive frame filtering.

In [ ]:
!colmap model_analyzer \
  --path "{SPARSE_DIR}/0"

## 15. Create Nerfstudio Dataset From Existing COLMAP Model

This converts the selected images and COLMAP sparse model into a Nerfstudio dataset. The --skip-colmap flag is intentional because COLMAP has already run manually with custom matching.

The local Nerfstudio dataset is synced back to Drive after creation. Downstream notebooks, including segmentation, use the Drive copy.

In [ ]:
NS_LOG = f"{DRIVE_PROJECT_DIR}/reports/ns_process_data_log.txt"

!rm -rf "{NS_DATA_DIR}"
!mkdir -p "{NS_DATA_DIR}"
!mkdir -p "$(dirname '{NS_LOG}')"

!ns-process-data images \
    --data "{IMAGE_DIR}" \
    --output-dir "{NS_DATA_DIR}" \
    --sfm-tool colmap \
    --colmap-model-path "{SPARSE_DIR}/0" \
    --skip-colmap \
    2>&1 | tee "$NS_LOG"

!rm -rf "{DRIVE_NS_DATA_DIR}"
!mkdir -p "{DRIVE_NS_DATA_DIR}"
!rsync -a --info=progress2 "{NS_DATA_DIR}/" "{DRIVE_NS_DATA_DIR}/"

## 16. Verify Nerfstudio Dataset Files

This confirms that the Nerfstudio output folder exists and that transforms.json was created.

In [ ]:
!ls -lh "{NS_DATA_DIR}"
!find "{NS_DATA_DIR}" -name "transforms.json"

## 17. Count Registered Frames

This opens transforms.json and prints the number of frames Nerfstudio will use. This count is more important than the number of images selected, because only registered frames contribute to the reconstruction and later segmentation alignment.

In [ ]:
import json

TRANSFORMS_PATH = f"{NS_DATA_DIR}/transforms.json"

with open(TRANSFORMS_PATH, "r") as f:
    data = json.load(f)

print("Registered frames:", len(data["frames"]))

## 18. Verify or Install CUDA-enabled PyTorch

This cell avoids uninstalling Torch unless CUDA is unavailable. On Colab T4, Torch is often already installed with CUDA support, so skipping unnecessary reinstall saves time and avoids version conflicts.

In [ ]:
import subprocess
import sys

try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

if not has_cuda:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ], check=True)
else:
    print("CUDA-enabled torch already available")

## 19. Confirm CUDA Availability

This prints the Torch version, whether CUDA is available, and the CUDA version visible to Torch. For Tesla T4 training, torch.cuda.is_available() should be True.

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

## 20. Train Gaussian Splat Model With Nerfstudio

This cell copies the Nerfstudio dataset to local /content storage, trains splatfacto, then syncs training outputs back to Drive.

### Training Knobs

- TRAIN_ITERATIONS controls runtime and quality. 3000 to 5000 is useful for iteration. 10000 or higher is better for final quality.
- TORCH_CUDA_ARCH_LIST="7.5" matches Tesla T4 and avoids unnecessary compilation targets.
- --downscale-factor 2 reduces memory and runtime. Lower values can improve detail but cost more GPU memory and time.

The nvidia-smi line verifies that Colab has actually assigned a GPU before training starts.

In [ ]:
# Local fast storage
LOCAL_NS_DATA="/content/nerfstudio-data-local"
LOCAL_OUT="/content/ns-training-local"

DRIVE_NS_DATA="/content/drive/MyDrive/room-model-project/scans/scan_001/nerfstudio-data"
DRIVE_OUT="/content/drive/MyDrive/room-model-project/scans/scan_001/training"
TRAIN_ITERATIONS=5000

!rm -rf "$LOCAL_NS_DATA" "$LOCAL_OUT"
!mkdir -p "$LOCAL_NS_DATA" "$LOCAL_OUT"

!rsync -a --info=progress2 "$DRIVE_NS_DATA/" "$LOCAL_NS_DATA/"
!nvidia-smi

!MAX_JOBS=1 TORCH_CUDA_ARCH_LIST="7.5" ns-train splatfacto \
  --max-num-iterations $TRAIN_ITERATIONS \
  --steps-per-save 500 \
  --vis tensorboard \
  --data "$LOCAL_NS_DATA" \
  --output-dir "$LOCAL_OUT" \
  nerfstudio-data \
  --downscale-factor 2 \
  2>&1 | tee "/content/ns_train_local_log.txt"
  

!mkdir -p "$DRIVE_OUT"
!mkdir -p "/content/drive/MyDrive/room-model-project/scans/scan_001/reports"

!rsync -a --info=progress2 "$LOCAL_OUT/" "$DRIVE_OUT/"
!cp /content/ns_train_local_log.txt \
   "/content/drive/MyDrive/room-model-project/scans/scan_001/reports/ns_train_local_log.txt"

## 21. Locate Training Checkpoints

This lists recent checkpoint files saved to Drive. Use it to confirm training completed and produced restorable checkpoints.

In [ ]:
!find "/content/drive/MyDrive/room-model-project/scans/scan_001" -name "*.ckpt" | sort | tail

## 22. Export Gaussian Splat

This finds the latest Nerfstudio config.yml and exports the trained Gaussian splat to the exports folder in Drive.

TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 is included for compatibility with model loading behavior in newer PyTorch versions.

In [ ]:
CONFIG_PATH = !find "/content/drive/MyDrive/room-model-project/scans/scan_001/training/nerfstudio-data-local/splatfacto" -name "config.yml" | sort | tail -1
CONFIG_PATH = CONFIG_PATH[0]

print(CONFIG_PATH)

EXPORT_DIR = "/content/drive/MyDrive/room-model-project/scans/scan_001/exports"

!rm -rf "{EXPORT_DIR}"
!mkdir -p "{EXPORT_DIR}"

!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat \
    --load-config "{CONFIG_PATH}" \
    --output-dir "{EXPORT_DIR}"
    

!find "{EXPORT_DIR}" -maxdepth 3 -type f | sort

## 23. Repair NumPy ABI Before Depth Rendering

This checks Nerfstudio in a fresh Python subprocess. If Colab has mixed binary wheels and raises `numpy.dtype size changed`, the cell repairs NumPy in the environment and rechecks in another subprocess. Because depth rendering also runs in a fresh subprocess, a notebook kernel restart is not required for this depth workflow.


In [ ]:
import sys
import subprocess

IMPORT_CHECK = "from nerfstudio.utils.eval_utils import eval_setup; import numpy as np; print('Nerfstudio import OK. NumPy:', np.__version__)"

def run_import_check():
    return subprocess.run(
        [sys.executable, '-c', IMPORT_CHECK],
        text=True,
        capture_output=True,
    )

result = run_import_check()
if result.returncode == 0:
    print(result.stdout.strip())
else:
    combined = (result.stdout or '') + (result.stderr or '')
    if 'numpy.dtype size changed' not in combined:
        print(combined)
        raise RuntimeError('Nerfstudio import check failed for a reason other than NumPy ABI mismatch.')

    print('Detected NumPy binary ABI mismatch in subprocess.')
    print('Repairing NumPy, then rechecking without restarting the notebook kernel...')
    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--force-reinstall',
        '--no-cache-dir',
        'numpy==1.26.4',
    ], check=True)

    result = run_import_check()
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError('Nerfstudio still fails to import after NumPy repair. Restart the kernel only if this second subprocess check fails.')
    print(result.stdout.strip())
    print('NumPy repair confirmed in a fresh Python subprocess. Continue to depth rendering.')


## 24. Render Metric Depth Maps

This cell first syncs the Drive Nerfstudio dataset into the local `/content/nerfstudio-data-local` path expected by the saved config. It uses `rsync` every time, so partial local copies are completed instead of being skipped. Depth rendering then runs in a fresh Python subprocess, which keeps package repairs isolated from the active notebook kernel.


In [ ]:
if not RUN_OPTIONAL_DEPTH_RENDERING:
    print('Optional depth rendering is disabled for the object-splat production path.')
    print('Set RUN_OPTIONAL_DEPTH_RENDERING=True and run this section only when depth maps are required.')
else:
    from pathlib import Path
    import os
    import subprocess
    import sys

    SCAN_DIR = Path('/content/drive/MyDrive/room-model-project/scans/scan_001')
    DRIVE_NS_DATA_DIR = SCAN_DIR / 'nerfstudio-data'
    LOCAL_NS_DATA_DIR = Path('/content/nerfstudio-data-local')
    DRIVE_TRAINING_DIR = SCAN_DIR / 'training'
    LOCAL_TRAINING_DIR = Path('/content/ns-training-local')
    CONFIG_ROOT = SCAN_DIR / 'training' / 'nerfstudio-data-local' / 'splatfacto'
    DEPTH_DIR = SCAN_DIR / 'depths'  # Backward-compatible primary depth folder used by pipeline.ipynb.
    DEPTH_VARIANTS_DIR = SCAN_DIR / 'depth_variants'
    DEPTH_CONFIDENCE_DIR = SCAN_DIR / 'depth_confidence'
    DEPTH_PREVIEW_DIR = SCAN_DIR / 'depth_previews_16bit_png'
    DEPTH_REPORT_PATH = SCAN_DIR / 'reports' / 'depth_render_summary.json'
    RENDER_SCRIPT = Path('/content/render_metric_depth_maps.py')
    DEPTH_RENDER_SCALE = 1.0  # Full resolution is preferred for 2D mask projection. Lower to 0.5/0.25 only if Colab OOMs.
    PRIMARY_DEPTH_PRIORITY = ['median_depth', 'expected_depth', 'depth', 'rendered_depth']
    CUDA_ARCH_LIST = '7.5'  # Tesla T4. Change only if using a different GPU architecture.
    TORCH_EXTENSIONS_DIR = Path('/content/torch_extensions_depth_render')

    configs = sorted(CONFIG_ROOT.rglob('config.yml'))
    assert configs, f'Missing trained Nerfstudio config under {CONFIG_ROOT}. Run training first.'
    CONFIG_PATH = configs[-1]
    print('CONFIG_PATH:', CONFIG_PATH)

    assert DRIVE_NS_DATA_DIR.exists(), f'Missing Drive Nerfstudio dataset: {DRIVE_NS_DATA_DIR}'
    assert DRIVE_TRAINING_DIR.exists(), f'Missing Drive training outputs: {DRIVE_TRAINING_DIR}'
    LOCAL_NS_DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_TRAINING_DIR.mkdir(parents=True, exist_ok=True)

    print('Syncing Nerfstudio dataset into local path expected by config...')
    subprocess.run([
        'rsync',
        '-ah',
        '--info=progress2',
        f'{DRIVE_NS_DATA_DIR}/',
        f'{LOCAL_NS_DATA_DIR}/',
    ], check=True)

    print('Syncing Nerfstudio training/checkpoints into local path expected by config...')
    subprocess.run([
        'rsync',
        '-ah',
        '--info=progress2',
        f'{DRIVE_TRAINING_DIR}/',
        f'{LOCAL_TRAINING_DIR}/',
    ], check=True)

    assert (LOCAL_NS_DATA_DIR / 'transforms.json').exists(), f'Missing local transforms.json: {LOCAL_NS_DATA_DIR / "transforms.json"}'
    checkpoint_dirs = sorted(LOCAL_TRAINING_DIR.rglob('nerfstudio_models'))
    assert checkpoint_dirs, f'Missing local checkpoint directories under {LOCAL_TRAINING_DIR}'
    print('Local checkpoint dirs found:', len(checkpoint_dirs))

    render_source_template = r'''
    from pathlib import Path
    import json
    import shutil
    import numpy as np
    import cv2
    import torch
    from tqdm.auto import tqdm
    from nerfstudio.utils.eval_utils import eval_setup

    CONFIG_PATH = Path({config_path_repr})
    DEPTH_DIR = Path({depth_dir_repr})
    DEPTH_VARIANTS_DIR = Path({depth_variants_dir_repr})
    DEPTH_CONFIDENCE_DIR = Path({depth_confidence_dir_repr})
    DEPTH_PREVIEW_DIR = Path({preview_dir_repr})
    DEPTH_REPORT_PATH = Path({report_path_repr})
    LOCAL_NS_DATA_DIR = Path({local_data_repr})
    DEPTH_RENDER_SCALE = {render_scale_repr}
    PRIMARY_DEPTH_PRIORITY = {primary_depth_priority_repr}
    DEPTH_LIKE_KEYS = ['median_depth', 'expected_depth', 'depth', 'rendered_depth']
    CONFIDENCE_LIKE_KEYS = ['accumulation', 'accumulation_map', 'alpha', 'opacity', 'weights_sum']
    LOW_CONFIDENCE_THRESHOLD = 0.05

    assert LOCAL_NS_DATA_DIR.exists(), f'Missing local Nerfstudio data: {{LOCAL_NS_DATA_DIR}}'
    assert (LOCAL_NS_DATA_DIR / 'transforms.json').exists(), f'Missing local transforms.json: {{LOCAL_NS_DATA_DIR / "transforms.json"}}'

    # Clear only this step's explicit outputs so stale frame files cannot contaminate projection matching.
    for directory in [DEPTH_DIR, DEPTH_VARIANTS_DIR, DEPTH_CONFIDENCE_DIR, DEPTH_PREVIEW_DIR]:
        shutil.rmtree(directory, ignore_errors=True)
        directory.mkdir(parents=True, exist_ok=True)
    DEPTH_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

    torch.set_grad_enabled(False)
    config, pipeline, checkpoint_path, step = eval_setup(CONFIG_PATH, test_mode='inference')
    pipeline.eval()
    device = pipeline.device
    dataset = pipeline.datamanager.train_dataset
    cameras = dataset.cameras
    image_filenames = [Path(p) for p in getattr(dataset, 'image_filenames', [])]

    def tensor_to_numpy(value):
        if value is None:
            return None
        if isinstance(value, (list, tuple)):
            if not value:
                return None
            value = value[0]
        if not hasattr(value, 'detach'):
            return None
        arr = value.detach().cpu().numpy().squeeze().astype(np.float32)
        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = arr[..., 0]
        return arr

    def sanitize_depth(depth):
        depth = np.asarray(depth, dtype=np.float32)
        if depth.ndim == 3 and depth.shape[-1] == 1:
            depth = depth[..., 0]
        depth[~np.isfinite(depth)] = 0.0
        depth = np.maximum(depth, 0.0)
        return depth

    def matching_map(outputs, keys, target_shape=None):
        for key in keys:
            if key not in outputs:
                continue
            arr = tensor_to_numpy(outputs[key])
            if arr is None:
                continue
            if arr.ndim == 3 and arr.shape[-1] == 1:
                arr = arr[..., 0]
            if arr.ndim != 2:
                continue
            if target_shape is not None and arr.shape != target_shape:
                continue
            return key, arr.astype(np.float32)
        return None, None

    def available_depth_outputs(outputs):
        result = {{}}
        for key in DEPTH_LIKE_KEYS:
            if key not in outputs:
                continue
            arr = tensor_to_numpy(outputs[key])
            if arr is None or arr.ndim != 2:
                continue
            result[key] = sanitize_depth(arr)
        return result

    def choose_primary_depth(depth_outputs):
        for key in PRIMARY_DEPTH_PRIORITY:
            if key in depth_outputs:
                return key, depth_outputs[key]
        if depth_outputs:
            key = sorted(depth_outputs.keys())[0]
            return key, depth_outputs[key]
        raise KeyError('No depth-like output found. Available priority keys: ' + ', '.join(PRIMARY_DEPTH_PRIORITY))

    def write_depth_preview(path, depth):
        valid = depth[np.isfinite(depth) & (depth > 0)]
        if valid.size:
            lo, hi = np.percentile(valid, [2, 98])
            normalized = np.clip((depth - lo) / max(hi - lo, 1e-6), 0, 1)
            preview = cv2.applyColorMap((normalized * 255).astype(np.uint8), cv2.COLORMAP_TURBO)
        else:
            preview = np.zeros((*depth.shape, 3), dtype=np.uint8)
        cv2.imwrite(str(path), preview)

    def depth_stats(depth):
        valid = depth[np.isfinite(depth) & (depth > 0)]
        return {{
            'valid_pixels': int(valid.size),
            'valid_ratio': float(valid.size / max(depth.size, 1)),
            'min_depth_m': float(valid.min()) if valid.size else None,
            'p01_depth_m': float(np.percentile(valid, 1)) if valid.size else None,
            'median_depth_m': float(np.percentile(valid, 50)) if valid.size else None,
            'p99_depth_m': float(np.percentile(valid, 99)) if valid.size else None,
            'max_depth_m': float(valid.max()) if valid.size else None,
        }}

    summary = {{
        'config_path': str(CONFIG_PATH),
        'checkpoint_path': str(checkpoint_path),
        'step': int(step) if step is not None else None,
        'primary_depth_dir': str(DEPTH_DIR),
        'depth_variants_dir': str(DEPTH_VARIANTS_DIR),
        'confidence_dir': str(DEPTH_CONFIDENCE_DIR),
        'preview_dir': str(DEPTH_PREVIEW_DIR),
        'render_scale': DEPTH_RENDER_SCALE,
        'primary_depth_priority': PRIMARY_DEPTH_PRIORITY,
        'low_confidence_threshold': LOW_CONFIDENCE_THRESHOLD,
        'frames': [],
    }}

    for i in tqdm(range(len(cameras)), desc='Rendering metric depth maps'):
        camera = cameras[i:i + 1].to(device)
        if DEPTH_RENDER_SCALE != 1.0:
            try:
                camera.rescale_output_resolution(scaling_factor=DEPTH_RENDER_SCALE)
            except TypeError:
                camera.rescale_output_resolution(DEPTH_RENDER_SCALE)

        outputs = pipeline.model.get_outputs_for_camera(camera)
        depth_outputs = available_depth_outputs(outputs)
        primary_key, primary_depth = choose_primary_depth(depth_outputs)
        confidence_key, confidence = matching_map(outputs, CONFIDENCE_LIKE_KEYS, target_shape=primary_depth.shape)

        if confidence is not None:
            confidence = np.clip(confidence.astype(np.float32), 0.0, 1.0)
            primary_depth = primary_depth.copy()
            primary_depth[confidence < LOW_CONFIDENCE_THRESHOLD] = 0.0

        if i < len(image_filenames):
            stem = image_filenames[i].stem
        else:
            stem = f'frame_{{i:05d}}'

        primary_npy_path = DEPTH_DIR / f'{{stem}}.npy'
        primary_png_path = DEPTH_PREVIEW_DIR / f'{{stem}}.png'
        np.save(primary_npy_path, primary_depth)
        write_depth_preview(primary_png_path, primary_depth)

        variant_records = {{}}
        for key, depth in depth_outputs.items():
            variant_dir = DEPTH_VARIANTS_DIR / key
            variant_dir.mkdir(parents=True, exist_ok=True)
            variant_path = variant_dir / f'{{stem}}.npy'
            np.save(variant_path, depth)
            variant_records[key] = {{
                'path': str(variant_path),
                'shape': list(depth.shape),
                **depth_stats(depth),
            }}

        confidence_path = None
        if confidence is not None:
            confidence_path = DEPTH_CONFIDENCE_DIR / f'{{stem}}.npy'
            np.save(confidence_path, confidence)

        summary['frames'].append({{
            'frame': stem,
            'camera_index': int(i),
            'image_filename': str(image_filenames[i]) if i < len(image_filenames) else None,
            'primary_depth_output_key': primary_key,
            'available_depth_output_keys': sorted(depth_outputs.keys()),
            'confidence_output_key': confidence_key,
            'primary_npy_path': str(primary_npy_path),
            'preview_png_path': str(primary_png_path),
            'confidence_path': str(confidence_path) if confidence_path is not None else None,
            'shape': list(primary_depth.shape),
            'render_scale': DEPTH_RENDER_SCALE,
            'variants': variant_records,
            **depth_stats(primary_depth),
        }})

        del outputs, camera
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    with open(DEPTH_REPORT_PATH, 'w') as f:
        json.dump(summary, f, indent=2)

    primary_key_counts = {{}}
    for frame in summary['frames']:
        primary_key_counts[frame['primary_depth_output_key']] = primary_key_counts.get(frame['primary_depth_output_key'], 0) + 1

    print('Rendered depth maps:', len(summary['frames']))
    print('Primary metric depth .npy dir:', DEPTH_DIR)
    print('Depth variants dir:', DEPTH_VARIANTS_DIR)
    print('Confidence dir:', DEPTH_CONFIDENCE_DIR)
    print('Preview dir:', DEPTH_PREVIEW_DIR)
    print('Primary depth key counts:', primary_key_counts)
    print('Depth render summary:', DEPTH_REPORT_PATH)
    '''

    render_source = render_source_template.format(
        config_path_repr=repr(str(CONFIG_PATH)),
        depth_dir_repr=repr(str(DEPTH_DIR)),
        depth_variants_dir_repr=repr(str(DEPTH_VARIANTS_DIR)),
        depth_confidence_dir_repr=repr(str(DEPTH_CONFIDENCE_DIR)),
        preview_dir_repr=repr(str(DEPTH_PREVIEW_DIR)),
        report_path_repr=repr(str(DEPTH_REPORT_PATH)),
        local_data_repr=repr(str(LOCAL_NS_DATA_DIR)),
        render_scale_repr=repr(float(DEPTH_RENDER_SCALE)),
        primary_depth_priority_repr=repr(PRIMARY_DEPTH_PRIORITY),
    )

    RENDER_SCRIPT.write_text(render_source)
    print('Running depth renderer subprocess:', RENDER_SCRIPT)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja'], check=True)
    if TORCH_EXTENSIONS_DIR.exists():
        import shutil
        shutil.rmtree(TORCH_EXTENSIONS_DIR, ignore_errors=True)
    TORCH_EXTENSIONS_DIR.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
    env['MAX_JOBS'] = '1'
    env['TORCH_CUDA_ARCH_LIST'] = CUDA_ARCH_LIST
    env['TORCH_EXTENSIONS_DIR'] = str(TORCH_EXTENSIONS_DIR)
    result = subprocess.run(
        [sys.executable, str(RENDER_SCRIPT)],
        text=True,
        capture_output=True,
        env=env,
    )

    log_path = SCAN_DIR / 'reports' / 'depth_renderer_subprocess.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(
        'STDOUT\n======\n' + (result.stdout or '') + '\n\nSTDERR\n======\n' + (result.stderr or '')
    )

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('Depth renderer log:', log_path)

    if result.returncode != 0:
        if result.returncode == -9:
            raise RuntimeError(f'Depth renderer was killed with exit code -9, likely due to RAM/GPU memory pressure. Lower DEPTH_RENDER_SCALE to 0.5 or 0.25 and rerun. See log: {log_path}')
        raise RuntimeError(f'Depth renderer subprocess failed with exit code {result.returncode}. See log: {log_path}')


## 25. Verify Depth Outputs

This sanity check confirms that metric `.npy` depth maps exist and prints a compact summary of the first few frames.

In [ ]:
from pathlib import Path
import json
import numpy as np

SCAN_DIR = Path('/content/drive/MyDrive/room-model-project/scans/scan_001')
DEPTH_DIR = SCAN_DIR / 'depths'
DEPTH_VARIANTS_DIR = SCAN_DIR / 'depth_variants'
DEPTH_CONFIDENCE_DIR = SCAN_DIR / 'depth_confidence'
DEPTH_REPORT_PATH = SCAN_DIR / 'reports' / 'depth_render_summary.json'

primary_depth_files = sorted(DEPTH_DIR.glob('*.npy'))
print('Primary depth .npy files:', len(primary_depth_files))
for path in primary_depth_files[:5]:
    depth = np.load(path)
    valid = depth[np.isfinite(depth) & (depth > 0)]
    print(path.name, 'shape=', depth.shape, 'valid=', valid.size, 'min_m=', float(valid.min()) if valid.size else None, 'median_m=', float(np.median(valid)) if valid.size else None, 'max_m=', float(valid.max()) if valid.size else None)

variant_dirs = sorted([p for p in DEPTH_VARIANTS_DIR.glob('*') if p.is_dir()])
print('Depth variant dirs:', [p.name for p in variant_dirs])
for variant_dir in variant_dirs:
    files = sorted(variant_dir.glob('*.npy'))
    print(variant_dir.name, 'files=', len(files))

confidence_files = sorted(DEPTH_CONFIDENCE_DIR.glob('*.npy')) if DEPTH_CONFIDENCE_DIR.exists() else []
print('Confidence maps:', len(confidence_files))

if DEPTH_REPORT_PATH.exists():
    with open(DEPTH_REPORT_PATH, 'r') as f:
        report = json.load(f)
    frames = report.get('frames', [])
    print('Report frames:', len(frames))
    print('Render scale:', report.get('render_scale'))
    print('Primary depth priority:', report.get('primary_depth_priority'))
    primary_counts = {}
    for frame in frames:
        key = frame.get('primary_depth_output_key')
        primary_counts[key] = primary_counts.get(key, 0) + 1
    print('Primary depth key counts:', primary_counts)
    if frames:
        print('First frame summary:', {k: frames[0].get(k) for k in ['frame', 'primary_depth_output_key', 'available_depth_output_keys', 'confidence_output_key', 'valid_ratio', 'median_depth_m']})
    print('Report path:', DEPTH_REPORT_PATH)

assert primary_depth_files, f'No primary depth files found in {DEPTH_DIR}'
